In [ ]:
import dask.dataframe as dd
import os

def load_all_data_dask(base_dir):
    all_data = []
    for year_folder in os.listdir(base_dir):
        if not year_folder.startswith('year='):
            continue
        
        year_dir = os.path.join(base_dir, year_folder)
        year_files = [f for f in os.listdir(year_dir) if f.endswith('.parquet')]

        for file in year_files:
            file_path = os.path.join(year_dir, file)
            try:
                df = dd.read_parquet(file_path)
                df['year'] = int(year_folder.split('=')[1])
                all_data.append(df)
            except Exception as e:
                print(f"Skipping file {file_path} due to error: {e}")
    
    if not all_data:
        raise ValueError("No data files were successfully loaded.")
    
    full_df = dd.concat(all_data, axis=0)
    return full_df

df_all = load_all_data_dask("../../preprocessed")


In [ ]:
import pandas as pd

df = df_all.copy()
if 'loan_status_binary' not in df.columns:
    if 'loan_status' in df.columns:
        df['loan_status_binary'] = df['loan_status'].apply(lambda x: 1 if x == 'Default' else 0, meta=('loan_status', 'int'))
        print("Created 'loan_status_binary' from 'loan_status'.")
    else:
        print("Warning: Neither 'loan_status_binary' nor 'loan_status' found.")

numerical_features = df.select_dtypes(include=['float64', 'int64']).columns


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'loan_amnt' in df.columns:
    df['loan_amnt'].compute().plot(kind='hist', ax=axes[0], color=sns.color_palette("viridis")[1], bins=50)
    axes[0].set_title('Loan Amount Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Loan Amount', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)

if 'int_rate' in df.columns:
    df['int_rate'].compute().plot(kind='hist', ax=axes[1], color=sns.color_palette("viridis")[1], bins=50)
    axes[1].set_title('Interest Rate Distribution', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Interest Rate', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, col in enumerate(numerical_features):
    if idx < len(axes):
        df[col].compute().plot(kind='hist', ax=axes[idx], bins=50, alpha=0.7, 
                               color=sns.color_palette("viridis")[1], edgecolor='black')
        axes[idx].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel(col, fontsize=10)
        axes[idx].set_ylabel('Frequency', fontsize=10)
        axes[idx].grid(alpha=0.3)

for idx in range(len(numerical_features), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
if 'loan_status_binary' in df.columns:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.ravel()

    for idx, col in enumerate(numerical_features):
        if idx < len(axes):
            df[[col, 'loan_status_binary']].compute().boxplot(column=col, by='loan_status_binary', ax=axes[idx])
            axes[idx].set_title(f'{col} by Loan Status', fontsize=12, fontweight='bold')
            axes[idx].set_xlabel('Loan Status (0=Paid, 1=Default)', fontsize=10)
            axes[idx].set_ylabel(col, fontsize=10)

    for idx in range(len(numerical_features), len(axes)):
        axes[idx].axis('off')

    plt.suptitle('')
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np

if len(numerical_features) > 0:
    corr_matrix = df[numerical_features].compute().corr()
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    fig_size = max(10, len(numerical_features) * 0.8)
    plt.figure(figsize=(fig_size, fig_size))
    
    sns.heatmap(
        corr_matrix, 
        mask=mask,
        annot=True, 
        fmt=".2f", 
        cmap="coolwarm", 
        square=True, 
        linewidths=0.5, 
        cbar_kws={"shrink": 0.8},
        annot_kws={"size": 9}
    )
    
    # Improve label readability
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    
    plt.title("Feature Correlation Matrix", fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
